In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from src.amp_colabfold.curation import (
    fetch_dramp, fetch_dbaasp,
    filter_sequences, cluster_cdhit_python, save_outputs,
    RAW_DIR, PROCESSED_DIR
)
import pandas as pd
print("Imports OK")
print(f"Raw data dir:       {RAW_DIR}")
print(f"Processed data dir: {PROCESSED_DIR}")

Imports OK
Raw data dir:       D:\Github_projects\amp-colabfold\data\raw
Processed data dir: D:\Github_projects\amp-colabfold\data\processed


In [2]:
df_dramp = fetch_dramp(save_path=RAW_DIR / "dramp_raw.csv")

# DBAASP API returns empty responses to bulk queries — skip for now
# We have 11,000+ sequences from DRAMP which is sufficient
df_all = df_dramp.copy()
print(f"\nCombined raw: {len(df_all)} sequences")
print(df_all["source"].value_counts())

Fetching DRAMP 4.0 ...
  DRAMP general fetch failed: 'ascii' codec can't decode byte 0xe2 in position 4: ordinal not in range(128)
  DRAMP antibacterial fetch failed: 'ascii' codec can't decode byte 0xe2 in position 3: ordinal not in range(128)
  DRAMP natural fetch failed: 'ascii' codec can't decode byte 0xce in position 13: ordinal not in range(128)
  DRAMP after dedup: 24076 unique sequences
  Saved → D:\Github_projects\amp-colabfold\data\raw\dramp_raw.csv

Combined raw: 24076 sequences
source
DRAMP4.0    24076
Name: count, dtype: int64


In [3]:
df_filtered = filter_sequences(df_all)
print(f"\nAfter filtering: {len(df_filtered)} sequences")
df_filtered["source"].value_counts()


Filtering 24076 total raw sequences ...
  After length filter (10–50 aa): 17696  (removed 6380)
  After canonical AA filter: 17037  (removed 659)
  After exact deduplication: 17037  (removed 0)

After filtering: 17037 sequences


source
DRAMP4.0    17037
Name: count, dtype: int64

In [4]:
df_curated = cluster_cdhit_python(df_filtered, identity=0.90)
print(f"\nFinal curated set: {len(df_curated)} sequences")
df_curated.head(10)


Clustering at 90% identity ...
  17000/17037 ...
  Representatives: 12435 / 17037

Final curated set: 12435 sequences


,id,name,sequence,activity,source
0,DRAMP00069,DRAMP00069,EYHLMNGANGYLTRVNGKTVYRVTKDPVSAVFGVISNCWGSAGAGF...,general,DRAMP4.0
1,DRAMP00416,DRAMP00416,KLCERSSGTWSGVCGNNNACKNQCIRLEGAQHGSCNYVFPAHKCIC...,general,DRAMP4.0
2,DRAMP18380,DRAMP18380,SGRGKTGGKARAKAKTRSSRAGLQFPVGRVHRLLRKGNYAQRVGAG...,general,DRAMP4.0
3,DRAMP02586,DRAMP02586,QRGGFTGPIPRPPPHGRPPLGPICNACYRLSFSDVRICCNFLGKCC...,general,DRAMP4.0
4,DRAMP03600,DRAMP03600,EFELDRICGYGTARCRKKCRSQEYRIGRCPNTYACCLRKWDESLLN...,general,DRAMP4.0
5,DRAMP00144,DRAMP00144,NRWTNAYSAALGCAVPGVKYGKKLGGVWGAVIGGVGGAAVCGLAGY...,general,DRAMP4.0
6,DRAMP00259,DRAMP00259,MQKPEIISADLGLCAVNEFVALAAIPGGAATFAVCQMPNLDEIVSN...,general,DRAMP4.0
7,DRAMP00397,DRAMP00397,KFCEKPSGTWSGVCGNSGACKDQCIRLEGAKHGSCNYKPPAHRCIC...,general,DRAMP4.0
8,DRAMP00419,DRAMP00419,RYCERSSGTWSGVCGNTDKCSSQCQRLEGAAHGSCNYVFPAHKCIC...,general,DRAMP4.0
9,DRAMP00429,DRAMP00429,LCNERPSQTWSGNCGNTAHCDKQCQDWEKASHGACHKRENHWKCFC...,general,DRAMP4.0


In [7]:
save_outputs(df_curated)
print("\nStage 1 complete ✓")


Saved metadata  → D:\Github_projects\amp-colabfold\data\processed\curated_amps_metadata.csv  (12435 sequences)


AttributeError: 'Pandas' object has no attribute 'get'

In [9]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

# add amp_id if not already present
if "amp_id" not in df_curated.columns:
    df_curated.insert(0, "amp_id", [f"AMP_{i:05d}" for i in range(len(df_curated))])

fasta_path = PROCESSED_DIR / "curated_amps.fasta"
records = [
    SeqRecord(
        Seq(row.sequence),
        id=row.amp_id,
        description=f"source={row.source} activity={getattr(row, 'activity', 'unknown')}",
    )
    for row in df_curated.itertuples()
]
SeqIO.write(records, fasta_path, "fasta")

# also re-save the CSV with amp_id included
df_curated.to_csv(PROCESSED_DIR / "curated_amps_metadata.csv", index=False)

print(f"Saved FASTA     → {fasta_path}")
print(f"Saved metadata  → {PROCESSED_DIR / 'curated_amps_metadata.csv'}")
print(f"Total sequences: {len(records)}")
print("\nStage 1 complete ✓")

Saved FASTA     → D:\Github_projects\amp-colabfold\data\processed\curated_amps.fasta
Saved metadata  → D:\Github_projects\amp-colabfold\data\processed\curated_amps_metadata.csv
Total sequences: 12435

Stage 1 complete ✓


## Stage 1 results

| Step | Input | Output | Removed |
|------|-------|--------|---------|
| Raw download (DRAMP 4.0) | — | 24,076 | — |
| Length filter (10–50 aa) | 24,076 | 17,696 | 6,380 |
| Canonical AA filter | 17,696 | 17,037 | 659 |
| Exact deduplication | 17,037 | 17,037 | 0 |
| 90% identity clustering | 17,037 | 12,435 | 4,602 |

**Final curated set: 12,435 sequences**
Outputs: `data/processed/curated_amps_metadata.csv`, `data/processed/curated_amps.fasta`

In [ ]:
import plotly.express as px

fig = px.histogram(
    df_curated,
    x=df_curated["sequence"].str.len(),
    color="source",
    nbins=40,
    labels={"x": "Peptide length (aa)", "count": "Count"},
    title="Curated AMP length distribution by source database",
    template="simple_white",
)
fig.update_layout(bargap=0.1)
fig.show()